# 18 SPPNet 与空间金字塔池化

前面学过卷积、池化、Flatten、全连接分类头和目标检测。这一课把它们连起来：不同大小的特征图怎样进入同一个全连接层？SPPNet 怎样减少检测中的重复卷积？

## 1. 为什么需要 SPP

假设 CNN 的卷积部分对两张不同大小的图片分别输出 $256 \times 13 \times 13$ 和 $256 \times 20 \times 20$ 的特征图。直接 Flatten 后，长度分别为 43264 和 102400。

同一个全连接层的输入维度在建立时固定，不能同时接收这两种长度。因此传统 CNN 常把图片缩放或裁剪到固定大小。

卷积层本身通常不要求固定的高和宽；这里的矛盾是**变化的特征图尺寸与固定输入维度的全连接层**。具体网络仍可能有最小输入尺寸等限制。

## 2. SPPNet 是什么

SPPNet 是使用**空间金字塔池化**（Spatial Pyramid Pooling，简称 SPP）的网络。在分类网络中，它位于卷积特征图和全连接层之间：

图片 → 卷积特征提取 → SPP → 固定长度向量 → 全连接层 → 分类结果

核心想法：无论输入特征图的高和宽怎样变化，按照预先指定的多个网格层级汇总，最终得到固定数量的数值。

“金字塔”表示从粗到细的多个网格层级，不是把图片真的堆成金字塔。

## 3. 先搞懂：一个网格为什么得到一个数

把一张特征图想成一张由数字组成的表。SPP 先在这张表上画网格，然后**对每个格子里的数做一次池化**。

例如使用最大池化：一个格子里无论有 4 个数还是 25 个数，最后都只取其中最大的**一个数**。所以：

$$
\text{输出数的个数} = \text{网格格子的个数}
$$

注意：池化之前，一个格子里通常包含特征图上的多个位置；池化之后，每个格子只留下一个代表值。固定输出数量的关键就在这里。

## 4. 用 4 × 4 特征图逐格计算

先只看一个通道，特征图是：

$$
\begin{bmatrix}
1 & 2 & 3 & 4 \\
5 & 6 & 7 & 8 \\
9 & 10 & 11 & 12 \\
13 & 14 & 15 & 16
\end{bmatrix}
$$

### 第一层：1 × 1 网格

整张图作为 1 个格子，从中取最大值，得到 $[16]$。因此输出有 $1 \times 1 = 1$ 个数。

### 第二层：2 × 2 网格

把**同一张特征图**划成 2 行、2 列，共 4 个格子。每个格子内部是一个 $2 \times 2$ 小区域：

| 区域 | 区域内的数 | 最大值 |
|---|---|---|
| 左上 | 1、2、5、6 | 6 |
| 右上 | 3、4、7、8 | 8 |
| 左下 | 9、10、13、14 | 14 |
| 右下 | 11、12、15、16 | 16 |

这一层得到 $[6, 8, 14, 16]$，共 $2 \times 2 = 4$ 个数。

注意：$2 \times 2$ 在这里指**划分后网格的行数和列数**。恰好对于本例的 $4 \times 4$ 特征图，每个格子内部也是 $2 \times 2$；换一张更大的特征图，格子内部大小就会改变。

## 5. 为什么叫“金字塔”，为什么要拼接

上面两个层级是对**同一张输入特征图分别划分**，而不是先得到 1 × 1，再把这个结果分成 2 × 2。

- 1 × 1：1 个大格子，得到整体的 1 个最大值。
- 2 × 2：4 个较小格子，得到 4 个位置各自的最大值。

从粗到细的多个网格层级组成“空间金字塔”。把不同层级的输出按约定顺序展平并拼接：

$$
[16] \;\Vert\; [6, 8, 14, 16] = [16, 6, 8, 14, 16]
$$

这里 $\Vert$ 表示拼接。因此每个通道一共得到 $1 + 4 = 5$ 个数。整体层级能概括全局，较细层级能区分大致位置；使用多个层级是一种特征表示设计，不是说只有这一种划分方式才正确。

## 6. 换成 8 × 8，为什么仍然是 5 个数

假设现在有另一张单通道特征图，高和宽都是 8。仍然采用相同的 1 × 1、2 × 2 层级，并对每个格子取一个最大值：

| 层级 | 4 × 4 输入时 | 8 × 8 输入时 | 两种输入各输出 |
|---|---|---|---|
| 1 × 1 | 1 个格子，每格包含 4 × 4 个位置 | 1 个格子，每格包含 8 × 8 个位置 | 1 个数 |
| 2 × 2 | 4 个格子，每格包含 2 × 2 个位置 | 4 个格子，每格包含 4 × 4 个位置 | 4 个数 |
| 拼接 | 1 + 4 个数 | 1 + 4 个数 | 5 个数 |

**尺寸变了，每个格子覆盖的输入位置数变了；但网格数量没变，每格输出一个数，因此输出数量没变。**数值通常会不同，保持一致的是输出的长度。

展平只是把池化后固定数量的数排成一行；它本身并不会把不同数量的数变成一样多。真正保证数量固定的是前面的“固定网格数 + 每格输出一个数”。

## 7. 多个层级和多个通道怎么算

如果选用 1 × 1、2 × 2、4 × 4 三个层级，那么一个通道分别产生 1、4、16 个数：

$$
1 \times 1 + 2 \times 2 + 4 \times 4 = 21
$$

如果特征图有 $C$ 个通道，就对**每个通道分别**做同样的池化。每个通道 21 个数，全部拼接后就是：

$$
C \times 21
$$

例如 256 个通道，对应 $256 \times 21 = 5376$ 个数。只要通道数和网格层级固定，空间尺寸为 $256 \times 13 \times 13$ 与 $256 \times 20 \times 20$ 的输入都得到 5376 维向量。

输入的格子尺寸不能均分时，需要按具体实现取整，区域可能有重叠；关键仍是指定每层输出多少个格子。网络实际运行还受输入尺寸、显存和实现等限制。

## 8. 和普通池化、Flatten 的区别

普通池化通常指定窗口和步长，例如 $2 \times 2$ 窗口、步长 2。输入空间尺寸变化时，输出空间尺寸一般也会变化。

Flatten 只改变排列形状，不减少数据个数；不同大小的特征图直接 Flatten 后仍然长度不同。

SPP 按多个固定网格数量池化，再把结果拼成向量。它压缩空间信息，但通过多个层级保留一定的粗略位置信息，并不能保留每一个原始位置。

## 9. SPPNet 怎样用于目标检测

目标检测既要判断类别，也要判断位置。早期 R-CNN 先产生许多**候选区域**，再对每个区域分别运行 CNN。候选区域大量重叠时，会重复计算很多相似的卷积特征。

SPPNet 的检测流程可以这样理解：

1. 为原图生成候选区域。
2. 整张图片经过卷积网络，得到共享特征图。
3. 把每个候选区域对应到特征图上的位置。
4. 对该区域的特征使用 SPP，得到固定长度向量。
5. 用向量进行类别判断和边界框调整，再经过后处理得到结果。

这样，同一张图片的多个候选区域可以共享卷积特征，减少逐区域重复卷积。候选区域生成和逐区域后续处理仍然需要时间；SPPNet 也不等于后来的端到端单阶段 YOLO。

## 10. 和你学过的 YOLO 有什么关系

YOLOv5 中也有 SPP、SPPF 这样的名字，它们同样利用不同尺度的池化来汇集上下文，但结构和用途不能直接画等号。

本课的经典 SPP 将不同网格层级的结果拼成**固定长度向量**，便于接全连接层或处理候选区域；YOLOv5 的 SPP/SPPF 模块在特征图上做不同尺度的池化并沿通道拼接，输出仍是**特征图**，供后续检测网络使用。

所以看到相同的 “SPP” 名字时，先看它的输入、输出和所在位置。

## 11. 局限与理解边界

- SPP 让不同空间尺寸的特征图能生成固定长度表示，不意味着所有图片都能以任意尺寸组成一个普通 batch；尺寸、显存和具体网络实现仍有约束。
- 池化会丢失一部分精细空间信息，输出向量不能还原原特征图。
- 原始 SPPNet 检测流程包含候选区域生成等多个步骤，其检测训练中卷积层微调也有局限；后来 Fast R-CNN 进一步改进了训练流程。

## 12. 本课小结

1. 固定输入维度的全连接层与变化尺寸的特征图之间存在形状矛盾。
2. SPP 用多个固定网格层级分别池化，再拼接结果，得到固定长度向量。
3. 每个通道的输出数等于各层级网格数之和；总长度还要乘通道数。
4. 检测中的 SPPNet 共享整图卷积特征，再为各候选区域提取固定长度表示。

## 13. 自测

1. 为什么直接 Flatten 不足以解决不同大小特征图进入同一个全连接层的问题？
2. 单通道特征图采用 $1 \times 1$ 和 $3 \times 3$ 两个层级，会得到多少个数？如果有 64 个通道呢？
3. 为什么多个网格层级保留了一部分位置信息，却没有保留全部空间细节？
4. 在目标检测中，SPPNet 的卷积特征为什么能够复用？
5. 经典 SPP 层的输出与 YOLOv5 SPPF 的输出，在形状和用途上有什么区别？

<details><summary>查看第 2 题答案</summary>

单通道得到 $1 + 9 = 10$ 个数；64 个通道得到 $64 \times 10 = 640$ 个数。

</details>

参考：[SPPNet 原论文](https://arxiv.org/abs/1406.4729)；[作者项目页面](https://people.csail.mit.edu/kaiming/eccv14sppnet/index.html)。